In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
MDS Scatter Plots for Variables (OH / FP)
-----------------------------------------

sub 内の STEP3.2 (variables) の結果を使って
MDS (linear/nonlinear × top3/cumeig) ＋ NbClust クラスタ の散布図を
PNG / PDF 両方で保存する。

出力先:
  sub/plots_MDS_scatter_variables/<dataset>/<mds_mode>/

使用データ:
  cluster_labels_matrix_variables_<dataset>_<ts>.csv
  MDScoords_<mds_mode>_variables_<dataset>_<ts>.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================
# 設定
# ==========================
BASE = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/"
            "20251026_under_25clusters_program_and_result/for_checking_20251127/sub")

DATASETS = ["OH", "FP"]

MDS_MODES = [
    "linear_top3",
    "linear_cumeig",
    "nonlinear_top3",
    "nonlinear_cumeig"
]

INDEX_LIST = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]

OUT = BASE / "plots_MDS_scatter_variables"
OUT.mkdir(exist_ok=True, parents=True)


def load_mds_coords(dataset, mode):
    """MDS 座標ファイルを読み込む"""
    mds_dir = (BASE / "03_clustering_STEP3.2_signlessCorr" /
               "run_20251130_153500" / "variables" / dataset)
    pattern = f"MDScoords_{mode}_variables_{dataset}_*.csv"
    files = list(mds_dir.glob(pattern))
    if not files:
        print(f"[WARN] No MDS coords found: {pattern}")
        return None
    df = pd.read_csv(files[0], index_col=0)
    return df


def load_cluster_labels(dataset):
    """cluster_labels_matrix を読み込む"""
    lab_dir = (BASE / "03_clustering_STEP3.2_signlessCorr" /
               "run_20251130_153500" / "variables" / dataset)
    files = list(lab_dir.glob(f"cluster_labels_matrix_variables_{dataset}_*.csv"))
    if not files:
        print(f"[WARN] No cluster labels found for {dataset}")
        return None
    df = pd.read_csv(files[0])
    return df


def plot_one(df_plot, dataset, mode, index, outdir):
    """1つの散布図を作成して保存 (PNG/PDF)"""
    fig, ax = plt.subplots(figsize=(7, 6), dpi=300)

    sns.scatterplot(
        data=df_plot,
        x="MDS1", y="MDS2",
        hue="Cluster",
        palette="tab20",
        s=45,
        edgecolor="black",
        linewidth=0.3,
        ax=ax
    )

    ax.set_title(f"MDS ({mode}) — NbClust {index} — Variables ({dataset})", fontsize=12)
    ax.set_xlabel("MDS1")
    ax.set_ylabel("MDS2")

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="Cluster",
              borderaxespad=0.0)

    out_png = outdir / f"MDS_{dataset}_{mode}_{index}.png"
    out_pdf = outdir / f"MDS_{dataset}_{mode}_{index}.pdf"

    fig.tight_layout()
    fig.savefig(out_png)
    fig.savefig(out_pdf)
    plt.close(fig)


# ==========================
# メイン処理
# ==========================

for dataset in DATASETS:
    print(f"\n===== Dataset: {dataset} =====")

    # クラスタラベル
    lab = load_cluster_labels(dataset)
    if lab is None:
        continue

    # id 列名の統一
    if "id" not in lab.columns:
        lab = lab.rename(columns={lab.columns[0]: "id"})

    lab = lab.set_index("id")

    for mode in MDS_MODES:
        coords = load_mds_coords(dataset, mode)
        if coords is None:
            continue

        # 結合
        df = coords.copy()
        df.index.name = "id"

        # 25パターンのクラスタを割り当てる
        for index in INDEX_LIST:
            colname = f"{mode}_{index}"
            if colname not in lab.columns:
                print(f"[WARN] Column not found: {colname}")
                continue

            df_plot = df[["MDS1", "MDS2"]].copy()
            df_plot["Cluster"] = lab[colname].reindex(df_plot.index)

            outdir = OUT / dataset / mode
            outdir.mkdir(parents=True, exist_ok=True)

            plot_one(df_plot, dataset, mode, index, outdir)

print("\n🎉 全 MDS 散布図（PNG/PDF）を作成しました。")
print(f"出力先: {OUT}")